# Hanacham を Style-Bert-VITS2 で学習する（Colab 2026対応版）

Google Drive の `マイドライブ/Style-Bert-VITS2/Data/Hanacham` を使い、環境構築から JP-Extra モデルの学習までを上から順番に実行するノートブックです。学習結果は `マイドライブ/Style-Bert-VITS2/model_assets/Hanacham` に保存します。

## 実行前の確認

1. Colab の **ランタイム → ランタイムのタイプを変更** で GPU（T4 など）を選びます。
2. データフォルダに `raw/` と `esd.list` があることを確認します。
3. 自分の声、または学習と利用の明確な許可を得た声だけを使ってください。

**以前のノートブックのセルとは混ぜず、このファイルを新しくColabへアップロードして上から実行してください。** 公式ノートブックと同系統の Python 3.10.11 専用環境を作り、依存関係をStyle-Bert-VITS2 2.7.0公開時点に固定します。環境構築後の再起動は不要です。既存の前処理結果とモデルは削除せず、Drive 内の `retrain_backups/` に退避します。

## 0. GPU と環境を準備

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/Style-Bert-VITS2-2.7.0")
VENV_DIR = Path("/content/sbv2-py310")
PYTHON_BIN = VENV_DIR / "bin/python"

os.environ["PATH"] += ":/root/.cargo/bin:/root/.local/bin"
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
# 2.7.0公開直後より新しい依存パッケージを選ばないようにします。
os.environ["UV_EXCLUDE_NEWER"] = "2025-08-25T00:00:00Z"

# 失敗したコマンドがあれば、その場でセルを停止します。
subprocess.run(["nvidia-smi"], check=True)
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", "2.7.0", "--depth", "1",
        "https://github.com/litagin02/Style-Bert-VITS2.git", str(REPO_DIR),
    ], check=True)

# ColabではStyle-Bert-VITS2に元からあるワーカーなし実行を使います。
worker_init = REPO_DIR / "style_bert_vits2/nlp/japanese/pyopenjtalk_worker/__init__.py"
worker_source = worker_init.read_text(encoding="utf-8")
direct_mode_patch = """

# Colab compatibility: use the built-in no-worker fallback.
def initialize_worker(port: int = WORKER_PORT) -> None:
    return None
"""
if "Colab compatibility: use the built-in no-worker fallback." not in worker_source:
    worker_source += direct_mode_patch
worker_init.write_text(worker_source, encoding="utf-8")

# 公式Colabノートブックと同系統のPython 3.10.11環境を作ります。
if shutil.which("uv") is None:
    subprocess.run(
        ["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
        check=True,
    )
uv_bin = shutil.which("uv") or "/root/.local/bin/uv"
subprocess.run([uv_bin, "python", "install", "3.10.11"], check=True)
subprocess.run([uv_bin, "venv", str(VENV_DIR), "--python", "3.10.11", "--clear"], check=True)
# pyopenjtalkが必要とするpkg_resourcesを確実に含む版です。
subprocess.run([
    uv_bin, "pip", "install", "--python", str(PYTHON_BIN),
    "setuptools==69.5.1", "wheel==0.43.0", "--no-progress",
], check=True, env=os.environ.copy())
subprocess.run([
    uv_bin, "pip", "install", "--python", str(PYTHON_BIN),
    "-r", str(REPO_DIR / "requirements-colab.txt"), "--no-progress",
], check=True, env=os.environ.copy())
# ビルド依存による更新を防ぐため、最後にもう一度正確な版へ戻します。
subprocess.run([
    uv_bin, "pip", "install", "--python", str(PYTHON_BIN),
    "setuptools==69.5.1", "tensorboard", "soxr", "--no-progress",
], check=True, env=os.environ.copy())
# 初期化より前に、今回問題になった依存を個別確認します。
dependency_check = """
import pkg_resources
import pyopenjtalk
import torch
from transformers import DebertaV2Model
from torch.utils.tensorboard import SummaryWriter
print('pkg_resources OK')
print('PyOpenJTalk import OK')
print('TensorBoard import OK')
print('DebertaV2Model import OK')
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.version.cuda)
assert torch.cuda.is_available(), 'CUDAを利用できません'
"""
subprocess.run([str(PYTHON_BIN), "-c", dependency_check], cwd=REPO_DIR, check=True)
subprocess.run(
    [str(PYTHON_BIN), "initialize.py", "--skip_default_models"],
    cwd=REPO_DIR, check=True,
)

# CUDAとPyOpenJTalkワーカーをここで検査します。
healthcheck = """
import torch
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker
assert torch.cuda.is_available(), '専用環境からCUDAを利用できません'
pyopenjtalk_worker.initialize_worker()
result = pyopenjtalk_worker.run_frontend('こんにちは')
print('Python/依存関係/CUDA/PyOpenJTalk直接実行 OK')
print('GPU:', torch.cuda.get_device_name(0))
print('日本語解析結果数:', len(result))
"""
health_result = subprocess.run(
    [str(PYTHON_BIN), "-c", healthcheck], cwd=REPO_DIR,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(health_result.stdout)
if health_result.returncode != 0:
    raise RuntimeError("環境の疎通確認に失敗しました。上に表示されたログを確認してください。")
print("環境構築が完了しました。ランタイムを再起動せず、次へ進んでください。")

## 1. Google Drive を接続して設定

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/Style-Bert-VITS2-2.7.0")
PYTHON_BIN = Path("/content/sbv2-py310/bin/python")
DRIVE_ROOT = Path("/content/drive/MyDrive/Style-Bert-VITS2")
DATASET_ROOT = DRIVE_ROOT / "Data"
ASSETS_ROOT = DRIVE_ROOT / "model_assets"
MODEL_NAME = "Hanacham"
DATASET_DIR = DATASET_ROOT / MODEL_NAME

# T4 を想定した無難な初期値。メモリ不足時は 2、余裕があれば 4 を試せます。
BATCH_SIZE = 2
# 100件の学習データとバッチサイズ2で約4000ステップに相当します。
EPOCHS = 80
SAVE_EVERY_STEPS = 1000
USE_JP_EXTRA = True
NORMALIZE = False
TRIM = False
YOMI_ERROR = "skip"

assert REPO_DIR.is_dir(), "環境構築セルを先に実行してください。"
assert PYTHON_BIN.is_file(), "Python 3.10環境がありません。環境構築セルを先に実行してください。"

%cd /content/Style-Bert-VITS2-2.7.0
ASSETS_ROOT.mkdir(parents=True, exist_ok=True)
paths_yaml = f'dataset_root: "{DATASET_ROOT}"\nassets_root: "{ASSETS_ROOT}"\n'
(REPO_DIR / "configs/paths.yml").write_text(paths_yaml, encoding="utf-8")
gpu_name = subprocess.check_output(
    [str(PYTHON_BIN), "-c", "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"],
    text=True,
).strip()
print(f"GPU: {gpu_name}")
print(f"学習データ: {DATASET_DIR}")
print(f"学習結果: {ASSETS_ROOT / MODEL_NAME}")

## 2. 学習データを検査

`esd.list` は1行につき `音声への相対パス|話者名|JP|文字起こし` の4項目です。音声へのパスは `raw/` からの相対パスにします。拡張子が mp3 などでも、公式仕様に従い `esd.list` 側は `.wav` と記載できます。

In [ ]:
from collections import Counter

raw_dir = DATASET_DIR / "raw"
esd_path = DATASET_DIR / "esd.list"
assert raw_dir.is_dir(), f"raw フォルダがありません: {raw_dir}"
assert esd_path.is_file(), f"esd.list がありません: {esd_path}"

audio_extensions = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac"}
audio_files = [p for p in raw_dir.rglob("*") if p.is_file() and p.suffix.lower() in audio_extensions]
audio_by_stem = {}
for path in audio_files:
    relative = path.relative_to(raw_dir)
    audio_by_stem[str(relative.with_suffix(""))] = path

errors = []
languages = Counter()
speakers = Counter()
nonempty_lines = []
for line_number, raw_line in enumerate(esd_path.read_text(encoding="utf-8-sig").splitlines(), 1):
    line = raw_line.strip()
    if not line:
        continue
    nonempty_lines.append(line)
    parts = line.split("|", 3)
    if len(parts) != 4:
        errors.append(f"{line_number}行目: 4項目ではありません")
        continue
    audio_name, speaker, language, text = parts
    if str(Path(audio_name).with_suffix("")) not in audio_by_stem:
        errors.append(f"{line_number}行目: 対応する音声がありません ({audio_name})")
    if language not in {"JP", "EN", "ZH"}:
        errors.append(f"{line_number}行目: 言語IDが不正です ({language})")
    if not speaker.strip() or not text.strip():
        errors.append(f"{line_number}行目: 話者名または文字起こしが空です")
    languages[language] += 1
    speakers[speaker] += 1

assert nonempty_lines, "esd.list に有効な行がありません。"
assert not errors, "データ検査で問題が見つかりました:\n" + "\n".join(errors[:30])
if USE_JP_EXTRA:
    assert set(languages) == {"JP"}, f"JP-Extra は日本語専用です。検出した言語: {dict(languages)}"

print(f"音声ファイル数: {len(audio_files)}")
print(f"esd.list の行数: {len(nonempty_lines)}")
print(f"話者: {dict(speakers)}")
print(f"言語: {dict(languages)}")
print("データ形式に問題は見つかりませんでした。")

## 3. 前回の成果物をバックアップして再学習を準備

このセルは `raw/` と `esd.list` を残し、前処理済みデータ、チェックポイント、推論用モデルを時刻付きフォルダへ移動します。これにより、前回の続きではなく最初から学習します。

In [ ]:
from datetime import datetime
import shutil

backup_marker = Path("/content/.Hanacham_backup_done")
assert not backup_marker.exists(), "このランタイムではバックアップ済みです。このセルを二重実行しないでください。"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = DRIVE_ROOT / "retrain_backups" / f"{MODEL_NAME}_{timestamp}"
dataset_backup = backup_dir / "Data" / MODEL_NAME
assets_backup = backup_dir / "model_assets" / MODEL_NAME

# 入力データ以外で Style-Bert-VITS2 が生成する項目だけを退避します。
generated_names = [
    "wavs", "models", "config.json", "train.list", "val.list",
    "esd.list.cleaned", "text_error.log",
]
moved = []
for name in generated_names:
    source = DATASET_DIR / name
    if source.exists():
        dataset_backup.mkdir(parents=True, exist_ok=True)
        shutil.move(str(source), str(dataset_backup / name))
        moved.append(str(source))

old_assets = ASSETS_ROOT / MODEL_NAME
if old_assets.exists():
    assets_backup.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(old_assets), str(assets_backup))
    moved.append(str(old_assets))

if moved:
    print("前回の成果物を退避しました:")
    print(f"  保存先: {backup_dir}")
    for item in moved:
        print(f"  - {item}")
else:
    print("退避対象はありません。新規学習として進めます。")
backup_marker.write_text(str(backup_dir), encoding="utf-8")

## 4. 前処理

音声の 44.1 kHz への変換、テキスト処理、BERT 特徴量・スタイル特徴量の生成をまとめて行います。読みに変換できない行は `YOMI_ERROR = "skip"` により除外され、詳細はデータフォルダの `text_error.log` に記録されます。

In [ ]:
command = [
    str(PYTHON_BIN), "preprocess_all.py",
    "--model_name", MODEL_NAME,
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--save_every_steps", str(SAVE_EVERY_STEPS),
    "--num_processes", "2",
    "--val_per_lang", "0",
    "--log_interval", "200",
    "--yomi_error", YOMI_ERROR,
]
if USE_JP_EXTRA:
    command.append("--use_jp_extra")
if NORMALIZE:
    command.append("--normalize")
if TRIM:
    command.append("--trim")
preprocess_env = os.environ.copy()
preprocess_env["MPLBACKEND"] = "Agg"
subprocess.run(command, cwd=REPO_DIR, check=True, env=preprocess_env)

train_list = DATASET_DIR / "train.list"
train_rows = [line for line in train_list.read_text(encoding="utf-8").splitlines() if line.strip()] if train_list.is_file() else []
bert_files = list((DATASET_DIR / "wavs").rglob("*.bert.pt"))
style_files = list((DATASET_DIR / "wavs").rglob("*.npy"))
assert train_rows, "前処理対象が0件です。text_error.logを確認してください。"
assert bert_files, "BERT特徴量が1件も生成されませんでした。"
assert style_files, "スタイル特徴量が1件も生成されませんでした。"
print(f"前処理が完了しました: 学習 {len(train_rows)}件 / BERT {len(bert_files)}件 / スタイル {len(style_files)}件")

## 5. 学習設定を確定

In [ ]:
dataset_path = str(DATASET_DIR)
config_path = str(DATASET_DIR / "config.json")
configure_code = """
import sys, yaml
from pathlib import Path
repo_dir, model_name = Path(sys.argv[1]), sys.argv[2]
with (repo_dir / 'default_config.yml').open(encoding='utf-8') as file:
    config = yaml.safe_load(file)
config['model_name'] = model_name
with (repo_dir / 'config.yml').open('w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True)
"""
subprocess.run([str(PYTHON_BIN), "-c", configure_code, str(REPO_DIR), MODEL_NAME], check=True)

assert Path(config_path).is_file(), f"config.json がありません: {config_path}"
train_rows = [line for line in (DATASET_DIR / "train.list").read_text(encoding="utf-8").splitlines() if line.strip()]
assert train_rows, "train.list が空です。前処理を確認してください。"
print(f"設定: {config_path}")
print(f"バッチサイズ: {BATCH_SIZE} / エポック数: {EPOCHS} / 保存間隔: {SAVE_EVERY_STEPS}")

## 6. 学習を開始

このセルは長時間かかります。Colab のタブを閉じず、接続が切れないようにしてください。CUDA メモリ不足になった場合は、ランタイムを再起動し、設定セルの `BATCH_SIZE` を 2 より小さくして前処理からやり直します。

In [ ]:
train_script = "train_ms_jp_extra.py" if USE_JP_EXTRA else "train_ms.py"
command = [
    str(PYTHON_BIN), train_script,
    "--config", config_path,
    "--model", dataset_path,
    "--assets_root", str(ASSETS_ROOT),
]
print("学習を開始します。")
result = subprocess.run(
    command, cwd=REPO_DIR, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f"学習に失敗しました（終了コード: {result.returncode}）")
print("学習が完了しました。")

## 7. 学習結果を確認

In [ ]:
asset_dir = ASSETS_ROOT / MODEL_NAME
model_files = sorted(asset_dir.glob("*.safetensors"), key=lambda p: p.stat().st_mtime)
required_files = [asset_dir / "config.json", asset_dir / "style_vectors.npy"]

assert model_files, f"safetensors が見つかりません: {asset_dir}"
for required_file in required_files:
    assert required_file.is_file(), f"必要なファイルがありません: {required_file}"

print("学習完了です。ローカルで使う3点:")
print(f"  モデル: {model_files[-1]}")
print(f"  設定: {asset_dir / 'config.json'}")
print(f"  スタイル: {asset_dir / 'style_vectors.npy'}")
print("Google Drive に保存済みなので、Colab から追加でダウンロードする必要はありません。")